In [1]:
import pandas as pd
import requests
import time
import random
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import json
import logging
from urllib.parse import quote
import re
import warnings
warnings.filterwarnings('ignore')

class RobustMenuImageCollector:
    def __init__(self, search_year=2024, search_quarter=1, target_images=1200):
        # 환경변수 강화된 검증
        self._validate_environment()
        
        self.search_year = search_year
        self.search_quarter = search_quarter
        self.search_months = self.get_quarter_months(search_quarter)
        self.target_images = target_images
        
        # API 엔드포인트들
        self.api_endpoints = {
            'news': "https://openapi.naver.com/v1/search/news.json",
            'blog': "https://openapi.naver.com/v1/search/blog.json", 
            'cafe': "https://openapi.naver.com/v1/search/cafearticle.json",
            'image': "https://openapi.naver.com/v1/search/image"
        }
        
        self.headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret,
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        # 할당 비율 (기본 20% + 비례 80%)
        self.base_allocation_ratio = 0.2
        self.proportional_allocation_ratio = 0.8
        
        # 수학적 균등 가중치
        self.api_weights = {
            'news': 0.82,   
            'blog': 0.04,   
            'cafe': 0.14    
        }
        
        # API 제한 설정
        self.api_limits = {
            'max_requests_per_day': 25000,
            'max_start_position': 1000,  # 네이버 API 제한
            'retry_count': 3,
            'base_delay': 0.1,
            'retry_delay_base': 2
        }
        
        # 결과 저장
        self.popularity_results = []
        self.menu_quotas = {}
        self.collected_images = []
        self.collected_urls = set()  # 중복 방지
        self.failed_requests = []  # 실패한 요청 추적
        
        self.daily_request_count = 0
        self.current_collected_count = 0
        
        # 로깅 설정
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        print(f"견고한 메뉴 이미지 수집 시스템 초기화 완료")
        print(f"분석 기간: {search_year}년 {search_quarter}분기")
        print(f"목표 이미지: {target_images}개")
        print(f"할당 비율: 기본 {self.base_allocation_ratio*100}% + 비례 {self.proportional_allocation_ratio*100}%")
    
    def _validate_environment(self):
        """환경변수 강화된 검증"""
        load_dotenv()
        
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        # 상세한 환경변수 검증
        if not self.client_id:
            raise ValueError(
                "Client_ID가 설정되지 않았습니다.\n"
                ".env 파일에 다음과 같이 설정해주세요:\n"
                "Client_ID=your_actual_client_id\n"
                "Client_Secret=your_actual_client_secret"
            )
        
        if not self.client_secret:
            raise ValueError(
                "Client_Secret이 설정되지 않았습니다.\n"
                ".env 파일에 다음과 같이 설정해주세요:\n"
                "Client_ID=your_actual_client_id\n"
                "Client_Secret=your_actual_client_secret"
            )
        
        # API 키 형식 기본 검증
        if len(self.client_id) < 10 or len(self.client_secret) < 10:
            self.logger.warning("API 키가 너무 짧습니다. 올바른 키인지 확인해주세요.")
        
        print("환경변수 검증 완료")
    
    def get_quarter_months(self, quarter):
        """분기별 월 정보 반환"""
        quarter_map = {
            1: [1, 2, 3], 2: [4, 5, 6], 
            3: [7, 8, 9], 4: [10, 11, 12]
        }
        return quarter_map.get(quarter, [1, 2, 3])
    
    def robust_api_call(self, api_type, query, start=1, display=100):
        """재시도 로직이 포함된 견고한 API 호출"""
        
        # API 제한 체크
        if self.daily_request_count >= self.api_limits['max_requests_per_day']:
            self.logger.error("일일 API 요청 한도에 도달했습니다.")
            return None
        
        # start 위치 제한 체크 (네이버 API 제한)
        if start > self.api_limits['max_start_position']:
            self.logger.warning(f"start 위치가 {self.api_limits['max_start_position']}을 초과했습니다.")
            return None
        
        api_url = self.api_endpoints.get(api_type)
        if not api_url:
            self.logger.error(f"지원하지 않는 API 타입: {api_type}")
            return None
        
        params = {
            'query': query,
            'start': start,
            'display': min(display, 100),
            'sort': 'date'
        }
        
        # 재시도 로직
        for attempt in range(self.api_limits['retry_count']):
            try:
                response = requests.get(
                    api_url, 
                    headers=self.headers, 
                    params=params, 
                    timeout=15
                )
                self.daily_request_count += 1
                
                if response.status_code == 200:
                    result = response.json()
                    if 'items' in result:
                        return result
                    else:
                        self.logger.warning(f"응답에 items가 없습니다: {api_type} - {query}")
                        return {'items': []}
                
                elif response.status_code == 429:
                    # Rate limit - 더 긴 대기
                    wait_time = (2 ** attempt) * 5  # 5, 10, 20초
                    self.logger.warning(f"API 제한 도달. {wait_time}초 대기... (시도 {attempt + 1}/{self.api_limits['retry_count']})")
                    time.sleep(wait_time)
                    continue
                
                elif response.status_code in [400, 401, 403]:
                    # 인증 오류 - 재시도 불가
                    self.logger.error(f"API 인증 오류 ({response.status_code}): {response.text}")
                    return None
                
                else:
                    # 기타 오류 - 재시도
                    self.logger.warning(f"API 오류 ({response.status_code}). 재시도 {attempt + 1}/{self.api_limits['retry_count']}")
                    
            except requests.exceptions.Timeout:
                self.logger.warning(f"API 타임아웃. 재시도 {attempt + 1}/{self.api_limits['retry_count']}")
                
            except requests.exceptions.RequestException as e:
                self.logger.warning(f"네트워크 오류: {e}. 재시도 {attempt + 1}/{self.api_limits['retry_count']}")
            
            # 재시도 전 대기 (지수 백오프)
            if attempt < self.api_limits['retry_count'] - 1:
                wait_time = (self.api_limits['retry_delay_base'] ** attempt) * random.uniform(0.5, 1.5)
                time.sleep(wait_time)
        
        # 모든 재시도 실패
        self.failed_requests.append({
            'api_type': api_type,
            'query': query,
            'start': start,
            'timestamp': datetime.now().isoformat()
        })
        self.logger.error(f"API 호출 최종 실패: {api_type} - {query}")
        return None
    
    def strict_date_validation(self, date_str, api_type):
        """엄격한 날짜 검증"""
        if not date_str or not isinstance(date_str, str):
            return False
        
        try:
            if api_type == 'news':
                # 뉴스: "Mon, 15 Jun 2024 09:30:00 +0900" 형식
                from email.utils import parsedate_tz
                parsed = parsedate_tz(date_str.strip())
                if parsed:
                    dt = datetime(*parsed[:6])
                    is_valid = dt.year == self.search_year and dt.month in self.search_months
                    return is_valid
            
            elif api_type in ['blog', 'cafe']:
                # 블로그/카페: "20240615" 형식
                if len(date_str) >= 8 and date_str.isdigit():
                    year = int(date_str[:4])
                    month = int(date_str[4:6])
                    is_valid = year == self.search_year and month in self.search_months
                    return is_valid
            
            elif api_type == 'image':
                # 이미지: pubDate 또는 URL 패턴 분석
                # 먼저 pubDate 시도
                if self.strict_date_validation(date_str, 'news'):
                    return True
                
                # URL 패턴 분석 (보조적)
                return self.extract_date_from_url(date_str)
            
        except (ValueError, IndexError, TypeError) as e:
            self.logger.debug(f"날짜 파싱 실패: {date_str} - {e}")
            return False
        
        return False
    
    def extract_date_from_url(self, url):
        """URL에서 날짜 추출 (보조적 방법)"""
        if not url:
            return False
        
        try:
            # 다양한 날짜 패턴
            patterns = [
                r'(\d{4})/(\d{1,2})/(\d{1,2})',  # 2024/1/15
                r'(\d{4})-(\d{1,2})-(\d{1,2})',  # 2024-1-15
                r'(\d{4})(\d{2})(\d{2})',        # 20240115
                r'(\d{4})/(\d{1,2})',            # 2024/1
                r'(\d{4})-(\d{1,2})',            # 2024-1
                r'(\d{4})(\d{2})',               # 202401
            ]
            
            for pattern in patterns:
                match = re.search(pattern, url)
                if match:
                    year = int(match.group(1))
                    month = int(match.group(2))
                    
                    if year == self.search_year and month in self.search_months:
                        return True
            
            return False
            
        except (ValueError, IndexError, AttributeError):
            return False
    
    def load_menu_data(self, csv_file_path):
        """메뉴 데이터 로드 with 검증"""
        try:
            if not os.path.exists(csv_file_path):
                raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {csv_file_path}")
            
            df = pd.read_csv(csv_file_path, encoding='utf-8')
            
            # 필수 컬럼 검증
            required_columns = ['대분류', '중분류', '소분류', '상세메뉴', '시각적특징']
            missing_columns = [col for col in required_columns if col not in df.columns]
            
            if missing_columns:
                raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")
            
            print(f"CSV 파일 로드 완료: {len(df)}개 행")
            
            menu_items = []
            for _, row in df.iterrows():
                detail_menus = [menu.strip() for menu in str(row['상세메뉴']).split(',') if menu.strip()]
                for detail_menu in detail_menus:
                    if detail_menu:  # 빈 문자열 제외
                        menu_items.append({
                            '대분류': row['대분류'],
                            '중분류': row['중분류'],
                            '소분류': row['소분류'],
                            '상세메뉴': detail_menu,
                            '시각적특징': row['시각적특징']
                        })
            
            print(f"총 {len(menu_items)}개의 개별 메뉴 항목 생성")
            return menu_items
            
        except Exception as e:
            self.logger.error(f"메뉴 데이터 로드 실패: {e}")
            raise
    
    def count_period_content_robust(self, api_type, menu_name):
        """견고한 콘텐츠 수 계산 (중복 제거 포함)"""
        unique_content_urls = set()  # 중복 제거용
        search_keywords = self.generate_api_keywords(menu_name, api_type)
        
        for keyword in search_keywords:
            start = 1
            max_pages = 2  # 테스트용 제한
            
            for page in range(max_pages):
                # start 위치 제한 체크
                if start > self.api_limits['max_start_position']:
                    break
                
                result = self.robust_api_call(api_type, keyword, start=start, display=100)
                
                if not result or 'items' not in result:
                    break
                
                items = result['items']
                if not items:
                    break
                
                valid_count_in_page = 0
                
                for item in items:
                    # API별 날짜 필드 추출
                    if api_type == 'news':
                        date_field = item.get('pubDate', '')
                        content_url = item.get('originallink', '') or item.get('link', '')
                    else:
                        date_field = item.get('postdate', '')
                        content_url = item.get('link', '')
                    
                    # 날짜 검증 및 중복 체크
                    if (self.strict_date_validation(date_field, api_type) and 
                        content_url and 
                        content_url not in unique_content_urls):
                        
                        unique_content_urls.add(content_url)
                        valid_count_in_page += 1
                
                # 해당 분기 콘텐츠가 없으면 중단 (날짜순 정렬)
                if valid_count_in_page == 0:
                    break
                
                start += len(items)
                time.sleep(self.api_limits['base_delay'])
        
        return len(unique_content_urls)
    
    def generate_api_keywords(self, menu_name, api_type):
        """API별 최적화된 검색 키워드 생성"""
        base_keywords = {
            'news': [menu_name, f"{menu_name} 맛집"],
            'blog': [menu_name, f"{menu_name} 후기"],
            'cafe': [menu_name, f"{menu_name} 추천"],
            'image': [menu_name, f"{menu_name} 음식", f"{menu_name} 요리"]
        }
        
        return base_keywords.get(api_type, [menu_name])
    
    def analyze_menu_popularity_robust(self, menu_item):
        """견고한 메뉴 인기도 분석"""
        menu_name = menu_item['상세메뉴']
        
        try:
            # 각 API에서 콘텐츠 수 계산
            news_count = self.count_period_content_robust('news', menu_name)
            blog_count = self.count_period_content_robust('blog', menu_name)
            cafe_count = self.count_period_content_robust('cafe', menu_name)
            
            # 수학적 균등 가중치 적용
            balanced_score = (
                news_count * self.api_weights['news'] +
                blog_count * self.api_weights['blog'] +
                cafe_count * self.api_weights['cafe']
            )
            
            return {
                'menu_name': menu_name,
                'news_count': news_count,
                'blog_count': blog_count,
                'cafe_count': cafe_count,
                'balanced_score': round(balanced_score, 2),
                'analysis_success': True
            }
            
        except Exception as e:
            self.logger.error(f"메뉴 '{menu_name}' 인기도 분석 실패: {e}")
            return {
                'menu_name': menu_name,
                'news_count': 0,
                'blog_count': 0,
                'cafe_count': 0,
                'balanced_score': 0.0,
                'analysis_success': False,
                'error_message': str(e)
            }
    
    def step1_analyze_popularity(self, csv_file_path):
        """1단계: 견고한 인기도 분석"""
        print("\n" + "="*50)
        print("1단계: 메뉴별 인기도 분석 시작")
        print("="*50)
        
        menu_items = self.load_menu_data(csv_file_path)
        successful_analyses = 0
        
        for i, menu_item in enumerate(menu_items):
            print(f"[{i+1}/{len(menu_items)}] {menu_item['상세메뉴']} 분석 중...")
            
            popularity_result = self.analyze_menu_popularity_robust(menu_item)
            popularity_result.update({
                '대분류': menu_item['대분류'],
                '중분류': menu_item['중분류'],
                '소분류': menu_item['소분류'],
                '시각적특징': menu_item['시각적특징']
            })
            
            self.popularity_results.append(popularity_result)
            
            if popularity_result['analysis_success']:
                successful_analyses += 1
                print(f"  균등화 점수: {popularity_result['balanced_score']:.2f}점")
            else:
                print(f"  분석 실패: {popularity_result.get('error_message', '알 수 없는 오류')}")
            
            # 진행상황 표시
            if (i + 1) % 10 == 0:
                progress = ((i + 1) / len(menu_items)) * 100
                success_rate = (successful_analyses / (i + 1)) * 100
                print(f"\n--- 진행률: {progress:.1f}% | 성공률: {success_rate:.1f}% ---")
            
            time.sleep(random.uniform(0.2, 0.5))
        
        print(f"\n1단계 완료!")
        print(f"성공한 분석: {successful_analyses}/{len(menu_items)}개")
        print(f"실패한 API 요청: {len(self.failed_requests)}개")
        print(f"총 API 요청: {self.daily_request_count}회")
        
        return menu_items
    
    def step2_calculate_quotas_precise(self, menu_items):
        """2단계: 정밀한 할당량 계산"""
        print("\n" + "="*50)
        print("2단계: 정밀한 할당량 계산")
        print("="*50)
        
        # 성공한 분석 결과만 사용
        successful_results = [r for r in self.popularity_results if r['analysis_success']]
        total_balanced_score = sum([r['balanced_score'] for r in successful_results])
        
        print(f"유효한 분석 결과: {len(successful_results)}개")
        print(f"총 균등화 점수: {total_balanced_score:.2f}점")
        
        # 정밀한 할당량 계산
        base_total = int(self.target_images * self.base_allocation_ratio)
        base_quota_per_menu = max(1, base_total // len(menu_items))
        used_base_total = base_quota_per_menu * len(menu_items)
        proportional_total = self.target_images - used_base_total
        
        print(f"기본 할당량: {base_quota_per_menu}개 × {len(menu_items)}메뉴 = {used_base_total}개")
        print(f"비례 할당량: {proportional_total}개")
        
        allocated_total = 0
        
        for result in self.popularity_results:
            menu_name = result['menu_name']
            balanced_score = result['balanced_score']
            
            # 기본 할당량
            base_quota = base_quota_per_menu
            
            # 비례 할당량 (성공한 경우만)
            if result['analysis_success'] and total_balanced_score > 0:
                ratio = balanced_score / total_balanced_score
                additional_quota = int(proportional_total * ratio)
            else:
                additional_quota = 0
            
            total_quota = base_quota + additional_quota
            self.menu_quotas[menu_name] = total_quota
            allocated_total += total_quota
        
        # 할당량 검증
        quota_difference = self.target_images - allocated_total
        if abs(quota_difference) > 0:
            print(f"할당량 차이: {quota_difference}개 (목표: {self.target_images}, 할당: {allocated_total})")
        
        # TOP 10 할당량 표시
        sorted_quotas = sorted(self.menu_quotas.items(), key=lambda x: x[1], reverse=True)
        print(f"\n할당량 TOP 10:")
        for i, (menu, quota) in enumerate(sorted_quotas[:10]):
            print(f"{i+1:2d}. {menu:<15}: {quota:3d}개")
        
        print(f"\n2단계 완료! 총 할당량: {allocated_total}개")
    
    def collect_menu_images_robust(self, menu_name, target_quota):
        """견고한 메뉴별 이미지 수집"""
        if target_quota <= 0:
            return []
        
        collected_images = []
        search_keywords = self.generate_api_keywords(menu_name, 'image')
        
        for keyword in search_keywords:
            if len(collected_images) >= target_quota:
                break
            
            start = 1
            max_pages = 3
            
            for page in range(max_pages):
                if len(collected_images) >= target_quota:
                    break
                
                # start 위치 제한 체크
                if start > self.api_limits['max_start_position']:
                    break
                
                result = self.robust_api_call('image', keyword, start=start, display=100)
                
                if not result or 'items' not in result:
                    break
                
                images = result['items']
                if not images:
                    break
                
                for img in images:
                    if len(collected_images) >= target_quota:
                        break
                    
                    img_url = img.get('link', '')
                    pub_date = img.get('pubDate', '')
                    
                    # 엄격한 검증: URL 중복 + 날짜 검증
                    if (img_url and 
                        img_url not in self.collected_urls and
                        (self.strict_date_validation(pub_date, 'image') or 
                         self.strict_date_validation(img_url, 'image'))):
                        
                        self.collected_urls.add(img_url)
                        
                        image_data = {
                            'menu_name': menu_name,
                            'image_url': img_url,
                            'title': img.get('title', '').replace('<b>', '').replace('</b>', ''),
                            'thumbnail': img.get('thumbnail', ''),
                            'size_width': img.get('sizewidth', ''),
                            'size_height': img.get('sizeheight', ''),
                            'pub_date': pub_date,
                            'search_keyword': keyword,
                            'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        }
                        
                        collected_images.append(image_data)
                
                start += len(images)
                time.sleep(self.api_limits['base_delay'])
        
        return collected_images
    
    def step3_collect_images_robust(self, menu_items):
        """3단계: 견고한 이미지 수집"""
        print("\n" + "="*50)
        print("3단계: 견고한 이미지 수집 시작")
        print("="*50)
        
        sorted_quotas = sorted(self.menu_quotas.items(), key=lambda x: x[1], reverse=True)
        successful_collections = 0
        
        for i, (menu_name, target_quota) in enumerate(sorted_quotas):
            if self.current_collected_count >= self.target_images:
                print(f"\n목표 수집량({self.target_images}개)에 도달하여 완료!")
                break
            
            print(f"[{i+1}/{len(sorted_quotas)}] '{menu_name}' 이미지 수집 (할당량: {target_quota}개)")
            
            remaining_quota = self.target_images - self.current_collected_count
            actual_quota = min(target_quota, remaining_quota)
            
            try:
                images = self.collect_menu_images_robust(menu_name, actual_quota)
                
                if images:
                    # 메뉴 정보 추가
                    menu_info = next((item for item in menu_items if item['상세메뉴'] == menu_name), {})
                    for img in images:
                        img.update({
                            '대분류': menu_info.get('대분류', ''),
                            '중분류': menu_info.get('중분류', ''),
                            '소분류': menu_info.get('소분류', ''),
                            '시각적특징': menu_info.get('시각적특징', ''),
                            '분석분기': f"{self.search_year}년 {self.search_quarter}분기"
                        })
                    
                    self.collected_images.extend(images)
                    self.current_collected_count += len(images)
                    successful_collections += 1
                    
                    success_rate = (len(images) / target_quota * 100) if target_quota > 0 else 0
                    print(f"  수집 완료: {len(images)}개 (달성률: {success_rate:.1f}%)")
                    print(f"  총 누적: {self.current_collected_count}/{self.target_images}개")
                else:
                    print(f"  수집된 이미지 없음")
                
            except Exception as e:
                self.logger.error(f"메뉴 '{menu_name}' 이미지 수집 실패: {e}")
                print(f"  수집 실패: {str(e)}")
            
            # 진행률 표시
            if (i + 1) % 20 == 0:
                overall_progress = (self.current_collected_count / self.target_images * 100)
                collection_success_rate = (successful_collections / (i + 1) * 100)
                print(f"\n--- 전체 진행률: {overall_progress:.1f}% | 수집 성공률: {collection_success_rate:.1f}% ---")
            
            time.sleep(random.uniform(0.2, 0.5))
        
        print(f"\n3단계 완료!")
        print(f"총 수집: {self.current_collected_count}개")
        print(f"성공한 수집: {successful_collections}/{len(sorted_quotas)}개 메뉴")
    
    def save_robust_results(self, output_file="robust_menu_images_2024Q1.xlsx"):
        """견고한 결과 저장"""
        try:
            print("\n결과 저장 중...")
            
            # 인기도 분석 결과
            popularity_df = pd.DataFrame(self.popularity_results)
            popularity_df['할당량'] = popularity_df['menu_name'].map(self.menu_quotas)
            popularity_df = popularity_df.sort_values('balanced_score', ascending=False)
            
            # 이미지 수집 결과
            if self.collected_images:
                images_df = pd.DataFrame(self.collected_images)
                
                # 메뉴별 수집 실적
                collection_stats = images_df.groupby('menu_name').size().reset_index(name='실제수집량')
                collection_stats = collection_stats.merge(
                    popularity_df[['menu_name', '할당량', 'balanced_score']], 
                    on='menu_name', 
                    how='right'
                ).fillna(0)
                collection_stats['달성률'] = (collection_stats['실제수집량'] / collection_stats['할당량'] * 100).round(1)
                collection_stats = collection_stats.sort_values('실제수집량', ascending=False)
            else:
                images_df = pd.DataFrame()
                collection_stats = pd.DataFrame()
            
            # 실패한 요청 통계
            failed_requests_df = pd.DataFrame(self.failed_requests) if self.failed_requests else pd.DataFrame()
            
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                # 인기도 분석 결과
                popularity_df.to_excel(writer, sheet_name='인기도분석결과', index=False)
                
                # 이미지 수집 결과
                if not images_df.empty:
                    images_df.to_excel(writer, sheet_name='수집된이미지', index=False)
                
                # 메뉴별 수집 실적
                if not collection_stats.empty:
                    collection_stats.to_excel(writer, sheet_name='메뉴별수집실적', index=False)
                
                # 실패한 요청들
                if not failed_requests_df.empty:
                    failed_requests_df.to_excel(writer, sheet_name='실패한요청', index=False)
                
                # 전체 요약
                months_str = f"{self.search_months[0]}-{self.search_months[-1]}월"
                successful_analyses = len([r for r in self.popularity_results if r['analysis_success']])
                
                summary_data = {
                    '항목': [
                        '분석 기간',
                        '총 메뉴 수',
                        '성공한 인기도 분석',
                        '실패한 인기도 분석',
                        '목표 이미지 수',
                        '실제 수집 이미지',
                        '목표 달성률',
                        '기본 할당 비율',
                        '비례 할당 비율',
                        'API 총 요청 횟수',
                        '실패한 API 요청',
                        '평균 메뉴당 수집',
                        '수집 완료 시각'
                    ],
                    '값': [
                        f"{self.search_year}년 {months_str}",
                        f"{len(self.popularity_results)}개",
                        f"{successful_analyses}개",
                        f"{len(self.popularity_results) - successful_analyses}개",
                        f"{self.target_images}개",
                        f"{len(self.collected_images)}개",
                        f"{(len(self.collected_images)/self.target_images*100):.1f}%",
                        f"{self.base_allocation_ratio*100}%",
                        f"{self.proportional_allocation_ratio*100}%",
                        f"{self.daily_request_count}회",
                        f"{len(self.failed_requests)}회",
                        f"{len(self.collected_images)/len(self.popularity_results):.1f}개" if self.popularity_results else "0개",
                        datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    ]
                }
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='전체요약', index=False)
            
            print(f"견고한 결과 저장 완료: {output_file}")
            print(f"수집된 이미지: {len(self.collected_images)}개")
            print(f"목표 달성률: {(len(self.collected_images)/self.target_images*100):.1f}%")
            print(f"실패한 API 요청: {len(self.failed_requests)}회")
            
        except Exception as e:
            self.logger.error(f"결과 저장 실패: {e}")
            print(f"결과 저장 중 오류 발생: {e}")
    
    def run_robust_process(self, csv_file_path):
        """견고한 통합 프로세스 실행"""
        try:
            print("견고한 메뉴 이미지 수집 프로세스 시작")
            print("="*60)
            
            # 1단계: 견고한 인기도 분석
            menu_items = self.step1_analyze_popularity(csv_file_path)
            
            # 2단계: 정밀한 할당량 계산  
            self.step2_calculate_quotas_precise(menu_items)
            
            # 3단계: 견고한 이미지 수집
            self.step3_collect_images_robust(menu_items)
            
            # 결과 저장
            self.save_robust_results()
            
            print("\n" + "="*60)
            print("견고한 프로세스 완료!")
            print(f"최종 수집: {len(self.collected_images)}개 이미지")
            print(f"목표 달성률: {(len(self.collected_images)/self.target_images*100):.1f}%")
            print(f"API 사용량: {self.daily_request_count}회")
            print(f"실패한 요청: {len(self.failed_requests)}회")
            print("="*60)
            
        except Exception as e:
            self.logger.error(f"견고한 프로세스 실행 중 치명적 오류: {e}")
            print(f"프로세스 실행 중 치명적 오류 발생: {e}")
            
            # 부분 결과라도 저장 시도
            if self.collected_images:
                try:
                    self.save_robust_results("partial_results.xlsx")
                    print("부분 결과를 저장했습니다: partial_results.xlsx")
                except:
                    print("부분 결과 저장도 실패했습니다.")
            
            import traceback
            traceback.print_exc()


def main():
    """메인 실행 함수"""
    CSV_FILE_PATH = "식당대12중53소132상세메뉴379분류.csv"
    SEARCH_YEAR = 2024
    SEARCH_QUARTER = 1
    TARGET_IMAGES = 8000  # 테스트용 1200 -> 실전용 목표량
    
    try:
        print("견고한 통합 메뉴 이미지 수집 시스템")
        print("="*50)
        
        # 견고한 시스템 초기화
        collector = RobustMenuImageCollector(
            search_year=SEARCH_YEAR,
            search_quarter=SEARCH_QUARTER,
            target_images=TARGET_IMAGES
        )
        
        # 파일 존재 확인
        if not os.path.exists(CSV_FILE_PATH):
            print(f"오류: CSV 파일을 찾을 수 없습니다 - {CSV_FILE_PATH}")
            print("현재 디렉토리의 파일들:")
            for file in os.listdir('.'):
                if file.endswith('.csv'):
                    print(f"  - {file}")
            return
        
        # .env 파일 확인
        if not os.path.exists('.env'):
            print("오류: .env 파일이 필요합니다.")
            print("다음 내용으로 .env 파일을 생성해주세요:")
            print("Client_ID=your_client_id")
            print("Client_Secret=your_client_secret")
            return
        
        # 견고한 프로세스 실행
        collector.run_robust_process(CSV_FILE_PATH)
        
    except Exception as e:
        print(f"메인 함수에서 오류 발생: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

견고한 통합 메뉴 이미지 수집 시스템
환경변수 검증 완료
견고한 메뉴 이미지 수집 시스템 초기화 완료
분석 기간: 2024년 1분기
목표 이미지: 8000개
할당 비율: 기본 20.0% + 비례 80.0%
견고한 메뉴 이미지 수집 프로세스 시작

1단계: 메뉴별 인기도 분석 시작
CSV 파일 로드 완료: 138개 행
총 381개의 개별 메뉴 항목 생성
[1/381] 제육볶음 분석 중...
  균등화 점수: 0.00점
[2/381] 매운제육볶음 분석 중...
  균등화 점수: 10.66점
[3/381] 두부제육볶음 분석 중...
  균등화 점수: 0.00점
[4/381] 된장찌개 분석 중...
  균등화 점수: 0.00점
[5/381] 김치찌개 분석 중...
  균등화 점수: 0.00점
[6/381] 청국장찌개 분석 중...
  균등화 점수: 0.00점
[7/381] 콩나물무침 분석 중...
  균등화 점수: 0.00점
[8/381] 시금치나물 분석 중...
  균등화 점수: 38.54점
[9/381] 도라지무침 분석 중...
  균등화 점수: 7.38점
[10/381] 계란말이 분석 중...
  균등화 점수: 0.00점

--- 진행률: 2.6% | 성공률: 100.0% ---
[11/381] 계란찜 분석 중...
  균등화 점수: 0.00점
[12/381] 스크램블에그 분석 중...
  균등화 점수: 6.56점
[13/381] 미역국 분석 중...
  균등화 점수: 0.00점
[14/381] 무국 분석 중...
  균등화 점수: 12.30점
[15/381] 콩나물국 분석 중...
  균등화 점수: 0.00점
[16/381] 부대찌개 분석 중...
  균등화 점수: 0.00점
[17/381] 햄부대찌개 분석 중...
  균등화 점수: 0.00점
[18/381] 치즈부대찌개 분석 중...
  균등화 점수: 0.00점
[19/381] 감자탕 분석 중...
  균등화 점수: 0.00점
[20/381] 뼈해장국 분석 중...
  균등화 점수: 0.00점

--- 진행

  균등화 점수: 59.04점
[189/381] 매운미소 분석 중...
  균등화 점수: 0.00점
[190/381] 츠케멘 분석 중...
  균등화 점수: 1.64점

--- 진행률: 49.9% | 성공률: 100.0% ---
[191/381] 아부라소바 분석 중...
  균등화 점수: 22.14점
[192/381] 탄탄멘 분석 중...
  균등화 점수: 3.28점
[193/381] 가츠동 분석 중...
  균등화 점수: 4.10점
[194/381] 규동 분석 중...
  균등화 점수: 36.08점
[195/381] 오야코동 분석 중...
  균등화 점수: 0.00점
[196/381] 텐동 분석 중...
  균등화 점수: 0.00점
[197/381] 가케우동 분석 중...
  균등화 점수: 2.46점
[198/381] 텐푸라우동 분석 중...
  균등화 점수: 0.82점
[199/381] 카레우동 분석 중...
  균등화 점수: 0.00점
[200/381] 자루소바 분석 중...
  균등화 점수: 1.64점

--- 진행률: 52.5% | 성공률: 100.0% ---
[201/381] 온소바 분석 중...
  균등화 점수: 0.00점
[202/381] 메밀소바 분석 중...
  균등화 점수: 0.00점
[203/381] 돈까스 분석 중...
  균등화 점수: 0.00점
[204/381] 치킨가츠 분석 중...
  균등화 점수: 2.46점
[205/381] 생선까스 분석 중...
  균등화 점수: 9.84점
[206/381] 새우텐푸라 분석 중...
  균등화 점수: 1.64점
[207/381] 야채텐푸라 분석 중...
  균등화 점수: 0.00점
[208/381] 모둠텐푸라 분석 중...
  균등화 점수: 0.00점
[209/381] 가라아게 분석 중...
  균등화 점수: 0.00점
[210/381] 치킨가라아게 분석 중...
  균등화 점수: 12.30점

--- 진행률: 55.1% | 성공률: 100.0% ---
[211/381] 닭튀김 분석 중...


[381/381] 그릭요거트 분석 중...
  균등화 점수: 0.00점

1단계 완료!
성공한 분석: 381/381개
실패한 API 요청: 0개
총 API 요청: 2438회

2단계: 정밀한 할당량 계산
유효한 분석 결과: 381개
총 균등화 점수: 1187.50점
기본 할당량: 4개 × 381메뉴 = 1524개
비례 할당량: 6476개
할당량 차이: 81개 (목표: 8000, 할당: 7919)

할당량 TOP 10:
 1. 마제소바           : 325개
 2. 마라파스타          : 263개
 3. 시금치나물          : 214개
 4. 규동             : 200개
 5. 삼선짬뽕           : 191개
 6. 타코야키           : 182개
 7. 탄탄면            : 147개
 8. 유니볶음면          : 133개
 9. 아부라소바          : 124개
10. 연어덮밥           : 120개

2단계 완료! 총 할당량: 7919개

3단계: 견고한 이미지 수집 시작
[1/379] '마제소바' 이미지 수집 (할당량: 325개)
  수집 완료: 7개 (달성률: 2.2%)
  총 누적: 7/8000개
[2/379] '마라파스타' 이미지 수집 (할당량: 263개)
  수집 완료: 25개 (달성률: 9.5%)
  총 누적: 32/8000개
[3/379] '시금치나물' 이미지 수집 (할당량: 214개)
  수집 완료: 16개 (달성률: 7.5%)
  총 누적: 48/8000개
[4/379] '규동' 이미지 수집 (할당량: 200개)
  수집 완료: 2개 (달성률: 1.0%)
  총 누적: 50/8000개
[5/379] '삼선짬뽕' 이미지 수집 (할당량: 191개)
  수집 완료: 6개 (달성률: 3.1%)
  총 누적: 56/8000개
[6/379] '타코야키' 이미지 수집 (할당량: 182개)
  수집 완료: 47개 (달성률: 25.8%)
  총 누적: 103/8000개
[7/379] 

  수집 완료: 21개 (달성률: 100.0%)
  총 누적: 1046/8000개
[97/379] '깐풍새우' 이미지 수집 (할당량: 17개)
  수집 완료: 3개 (달성률: 17.6%)
  총 누적: 1049/8000개
[98/379] '사천면' 이미지 수집 (할당량: 17개)
  수집 완료: 1개 (달성률: 5.9%)
  총 누적: 1050/8000개
[99/379] '마라룽샤' 이미지 수집 (할당량: 17개)
  수집 완료: 1개 (달성률: 5.9%)
  총 누적: 1051/8000개
[100/379] '참치사시미' 이미지 수집 (할당량: 17개)
  수집 완료: 11개 (달성률: 64.7%)
  총 누적: 1062/8000개

--- 전체 진행률: 13.3% | 수집 성공률: 98.0% ---
[101/379] '가케우동' 이미지 수집 (할당량: 17개)
  수집 완료: 3개 (달성률: 17.6%)
  총 누적: 1065/8000개
[102/379] '치킨가츠' 이미지 수집 (할당량: 17개)
  수집 완료: 2개 (달성률: 11.8%)
  총 누적: 1067/8000개
[103/379] '베이크드파스타' 이미지 수집 (할당량: 17개)
  수집 완료: 17개 (달성률: 100.0%)
  총 누적: 1084/8000개
[104/379] '그라탱' 이미지 수집 (할당량: 17개)
  수집 완료: 3개 (달성률: 17.6%)
  총 누적: 1087/8000개
[105/379] '바비큐립' 이미지 수집 (할당량: 17개)
  수집 완료: 4개 (달성률: 23.5%)
  총 누적: 1091/8000개
[106/379] '두꺼운피자' 이미지 수집 (할당량: 17개)
  수집 완료: 17개 (달성률: 100.0%)
  총 누적: 1108/8000개
[107/379] '카페프라페' 이미지 수집 (할당량: 17개)
  수집 완료: 17개 (달성률: 100.0%)
  총 누적: 1125/8000개
[108/379] '불닭파스타' 이미지 수집 (할당량: 17개)
  수집

2025-07-15 17:17:44,121 - WARNING - API 제한 도달. 5초 대기... (시도 1/3)


  수집 완료: 6개 (달성률: 75.0%)
  총 누적: 1303/8000개
[140/379] '렌당' 이미지 수집 (할당량: 8개)
  수집 완료: 5개 (달성률: 62.5%)
  총 누적: 1308/8000개

--- 전체 진행률: 16.4% | 수집 성공률: 96.4% ---
[141/379] '콤비네이션' 이미지 수집 (할당량: 8개)
  수집 완료: 8개 (달성률: 100.0%)
  총 누적: 1316/8000개
[142/379] '롱블랙' 이미지 수집 (할당량: 8개)
  수집 완료: 8개 (달성률: 100.0%)
  총 누적: 1324/8000개
[143/379] '인절미빙수' 이미지 수집 (할당량: 8개)
  수집 완료: 5개 (달성률: 62.5%)
  총 누적: 1329/8000개
[144/379] '라멘버거' 이미지 수집 (할당량: 8개)
  수집 완료: 3개 (달성률: 37.5%)
  총 누적: 1332/8000개
[145/379] '제육볶음' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1336/8000개
[146/379] '두부제육볶음' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1340/8000개
[147/379] '된장찌개' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1344/8000개
[148/379] '김치찌개' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1348/8000개
[149/379] '청국장찌개' 이미지 수집 (할당량: 4개)
  수집 완료: 2개 (달성률: 50.0%)
  총 누적: 1350/8000개
[150/379] '콩나물무침' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1354/8000개
[151/379] '계란말이' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성

  수집 완료: 1개 (달성률: 25.0%)
  총 누적: 1703/8000개
[242/379] '해물탕' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1707/8000개
[243/379] '탕수육' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1711/8000개
[244/379] '깐풍기' 이미지 수집 (할당량: 4개)
  수집 완료: 2개 (달성률: 50.0%)
  총 누적: 1713/8000개
[245/379] '깐쇼새우' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1717/8000개
[246/379] '칠리새우' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1721/8000개
[247/379] '유린기' 이미지 수집 (할당량: 4개)
  수집 완료: 2개 (달성률: 50.0%)
  총 누적: 1723/8000개
[248/379] '마라탕' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1727/8000개
[249/379] '마라샹궈' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1731/8000개
[250/379] '훠궈' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1735/8000개
[251/379] '마라두부' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1739/8000개
[252/379] '군만두' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1743/8000개
[253/379] '왕만두' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 1747/8000개
[254/379] '탕짜면' 이미지

[346/379] '카페라떼' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2076/8000개
[347/379] '마끼아또' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2080/8000개
[348/379] '바닐라라떼' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2084/8000개
[349/379] '카라멜마끼아또' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2088/8000개
[350/379] '헤이즐넛' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2092/8000개
[351/379] '얼그레이' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2096/8000개
[352/379] '레몬에이드' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2100/8000개
[353/379] '탄산음료' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2104/8000개
[354/379] '치즈케이크' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2108/8000개
[355/379] '초콜릿케이크' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2112/8000개
[356/379] '생크림케이크' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2116/8000개
[357/379] '크루아상' 이미지 수집 (할당량: 4개)
  수집 완료: 4개 (달성률: 100.0%)
  총 누적: 2120/8000개
[358/379] '베이글' 이미지 수집 (할당량: 4개)
  수집 완료: 